# Lab 2.3 (No Key) - Prompting Techniques on a Local Model (Solution)
**Day 2 - Session 3**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/gen-ai-companion-intuite/blob/main/day2/lab2_3_prompting_nokey/solution/lab2_3_prompting_nokey.ipynb)

**Goal:** Apply zero-shot, few-shot, and chain-of-thought prompting to sentiment, summarization, and reasoning, all on a small local model with no API key.

> **How this runs without an API key.** This version uses a small instruction-tuned model, `google/flan-t5-base`, that downloads once and runs on your machine (CPU is fine). No provider key and no shared client. The prompting techniques are identical to the API lab; only the call site changes. A small local model is weaker than a hosted model, so we use tasks it handles well (labeling, summarization, step-by-step reasoning).

This solution has the full working code.

In [ ]:
# Scoped install for this lab (safe to re-run; preinstalled on Colab).
%pip install -q transformers torch

## Setup: load a local instruction model
This downloads `flan-t5-base` once (about 1 GB) and caches it. After the first run it works offline.

In [ ]:
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

MODEL_NAME = "google/flan-t5-base"  # use "google/flan-t5-small" for a smaller download

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model.eval()


def generate(prompt, max_new_tokens=64):
    """Greedy text-in, text-out call to the local model (deterministic)."""
    inputs = tokenizer(prompt, return_tensors="pt")
    output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens)
    return tokenizer.decode(output_ids[0], skip_special_tokens=True)


print("Loaded", MODEL_NAME)

## Part 1. Sentiment: zero-shot vs few-shot
Classify each review as POSITIVE, NEGATIVE, or NEUTRAL. Few-shot examples pin down the label set and the exact output format.

In [ ]:
REVIEWS = [
    "The update completely broke my workflow.",
    "Best onboarding experience I have had.",
    "The battery could be better but I like it.",
]

# Zero-shot classification
for review in REVIEWS:
    prompt = (
        "Classify the sentiment as POSITIVE, NEGATIVE, or NEUTRAL. "
        f"Text: {review}"
    )
    label = generate(prompt)
    print(f"{label.strip():>9}  <-  {review}")

In [ ]:
# Few-shot: examples go inline in the prompt (T5 is not a chat model).
FEWSHOT = (
    "Classify the sentiment as POSITIVE, NEGATIVE, or NEUTRAL.\n"
    "Text: Best purchase ever. Sentiment: POSITIVE\n"
    "Text: Broke after one day. Sentiment: NEGATIVE\n"
    "Text: It arrived on time. Sentiment: NEUTRAL\n"
)

for review in REVIEWS:
    label = generate(FEWSHOT + f"Text: {review} Sentiment:")
    print(f"{label.strip():>9}  <-  {review}")

## Part 2. Summarization: zero-shot then refined
Start with a plain summary, then add a role and a format to tighten it.

In [ ]:
ARTICLE = (
    "Retrieval-augmented generation (RAG) combines a language model with a "
    "search step. Before answering, the system retrieves relevant documents "
    "from a knowledge base and adds them to the prompt. This grounds the "
    "answer in real data, reduces hallucination, and lets the model use "
    "private or up-to-date information it never saw during training."
)

zero_shot = generate(f"Summarize this text: {ARTICLE}", max_new_tokens=80)
print("Zero-shot summary:\n", zero_shot)

In [ ]:
refined = generate(
    f"Summarize the text in one sentence for a busy engineer: {ARTICLE}",
    max_new_tokens=80,
)
print("Refined summary:\n", refined)

## Part 3. Chain-of-thought
Ask the model to reason step by step. On a small model this can flip a wrong direct answer into a correct one.

In [ ]:
QUESTION = "A week has 7 days. How many days are in 3 weeks?"

direct = generate(QUESTION)
print("Direct:      ", direct)

stepwise = generate(QUESTION + " Think step by step.")
print("Step by step:", stepwise)

## Stretch goals
1. Translation with a local model.
2. Validate the label against the allowed set.
3. Note where a hosted model would beat this small one (for example code).

In [ ]:
print("Translate:", generate("Translate to French: The meeting is at noon."))

ALLOWED = {"POSITIVE", "NEGATIVE", "NEUTRAL"}
raw = generate(
    "Classify the sentiment as POSITIVE, NEGATIVE, or NEUTRAL. "
    f"Text: {REVIEWS[0]}"
)
label = raw.strip().upper()
print("Label valid?", label in ALLOWED, "->", label)

## Expected output
- Zero-shot and few-shot agree on the clear reviews (NEGATIVE, POSITIVE, POSITIVE).
- The summaries are short and on topic.
- The direct answer to the weeks question is often wrong; the step-by-step answer reaches 21.
- The stretch cell returns a French sentence and a valid label.